In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

print("Libraries imported successfully!")

Libraries imported successfully!


In [7]:
courses = pd.read_csv("courses.csv")
assessments = pd.read_csv("assessments.csv")
vle = pd.read_csv("vle.csv")
studentInfo = pd.read_csv("studentInfo.csv")
studentRegistration = pd.read_csv("studentRegistration.csv")
studentAssessment = pd.read_csv("studentAssessment.csv")
studentVle = pd.read_csv("studentVle.csv")

print("All datasets loaded successfully!")

All datasets loaded successfully!


In [8]:
# Map final result to binary classification (1 = Success, 0 = Fail/Withdraw)
studentInfo["target"] = studentInfo["final_result"].map({
    "Pass": 1,
    "Distinction": 1,
    "Fail": 0,
    "Withdrawn": 0
})

In [9]:
# Start with studentInfo as the base
ml_data = studentInfo.copy()

# 1. Merge Registration Data (FIXED: merging on module and presentation as well!)
ml_data = ml_data.merge(
    studentRegistration[['id_student', 'code_module', 'code_presentation', 'date_registration']],
    on=['id_student', 'code_module', 'code_presentation'],
    how='left'
)

# 2. Merge Course Data
ml_data = ml_data.merge(
    courses,
    on=['code_module', 'code_presentation'],
    how='left'
)

print(f"Final shape of preprocessed data: {ml_data.shape}")

Final shape of preprocessed data: (32593, 15)


In [10]:
# Save this clean, merged dataset so your next notebook can just load it directly!
ml_data.to_csv("preprocessed_oulad_data.csv", index=False)
print("Preprocessed data saved!")

Preprocessed data saved!


In [11]:
# === RUN THIS IN YOUR PREPROCESSING NOTEBOOK (NOTEBOOK 1) ===

import pandas as pd

# 1. Load the core datasets if you haven't already
studentInfo = pd.read_csv("studentInfo.csv")
studentRegistration = pd.read_csv("studentRegistration.csv")
courses = pd.read_csv("courses.csv")
studentVle = pd.read_csv("studentVle.csv")  # We need this for the clicks!

# 2. Create the target variable
studentInfo["target"] = studentInfo["final_result"].map({
    "Pass": 1, "Distinction": 1, "Fail": 0, "Withdrawn": 0
})

# 3. FEATURE ENGINEERING: Calculate clicks ONLY from the first 14 days of the course
# This completely avoids data leakage because it simulates what we know 2 weeks into the term
early_vle = studentVle[studentVle['date'] <= 14]

early_clicks = early_vle.groupby(['id_student', 'code_module', 'code_presentation'])['sum_click'].sum().reset_index()
early_clicks.rename(columns={'sum_click': 'clicks_first_14_days'}, inplace=True)

# 4. Merge everything together safely
ml_data = studentInfo.copy()

# Merge Registration
ml_data = ml_data.merge(
    studentRegistration[['id_student', 'code_module', 'code_presentation', 'date_registration']],
    on=['id_student', 'code_module', 'code_presentation'],
    how='left'
)

# Merge Courses
ml_data = ml_data.merge(courses, on=['code_module', 'code_presentation'], how='left')

# Merge our brand new Early Behavioral Feature!
ml_data = ml_data.merge(
    early_clicks, 
    on=['id_student', 'code_module', 'code_presentation'], 
    how='left'
)

# If a student had no clicks in the first 14 days, fill their missing click count with 0
ml_data['clicks_first_14_days'] = ml_data['clicks_first_14_days'].fillna(0)

# 5. Save the upgraded dataset
ml_data.to_csv("preprocessed_oulad_data.csv", index=False)
print("Upgraded dataset with early behavioral features saved successfully!")

Upgraded dataset with early behavioral features saved successfully!
